In [5]:
from datetime import datetime, timezone
from sqlalchemy.orm import sessionmaker
from event_clustering import get_engine, create_tables, Article, EventClustering
import warnings
warnings.filterwarnings("ignore")

***
## Подключение датасета с финансовыми статьями с Hugging Face, для проверки работы системы:

In [2]:
from datasets import load_dataset

ds = load_dataset("Kasymkhan/RussianFinancialNews")
train_data = ds["train"]

Using the latest cached version of the dataset since Kasymkhan/RussianFinancialNews couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at C:\Users\hp\.cache\huggingface\datasets\Kasymkhan___russian_financial_news\default\0.0.0\f83da5c05c82c9086806cf90ded4155052f9605c (last modified on Sun Dec 14 23:37:09 2025).


In [3]:
print(train_data.column_names)
for i in range(2):
    row = train_data[i]
    print("Row:", row)

['title', 'body', 'date', 'time', 'tags', 'source', '__index_level_0__']
Row: {'title': 'no title', 'body': 'Российские отели «всё включено» обяжут предлагать гостям отечественный алкоголь во время обедов и ужинов, объём будет неограничен', 'date': '2023-03-10', 'time': 'not stated', 'tags': 'no tags', 'source': 'smart_lab', '__index_level_0__': 68555}
Row: {'title': 'no title', 'body': '" Самолет (SMLT) в 4кв2023 собирается побить рекорд 3кв. Менеджмент подтверждает цель в 1 600 тыс. кв. м продаж за весь 2023 год. #аналитика #факты Видимо, менеджмент видит уже по октябрьским цифрам, что даже после роста ключевой ставки продажи на первичке идут хорошо. Первичный рынок, особенно массовый сегмент, на котором работает Самолет — бенефициар программ льготной ипотеки. Цель на год 1600 кв.м.: • по итогам 9 месяцев есть 973 тыс. кв. м. • консолидация МИЦ даст примерно 100 тыс. кв. м • таким образом, продажи Самолёта без МИЦ в 4 квартале могут составить порядка 527 тыс. кв. м - на 25% больше, ч

***
### Инициализация базы данных

На этом шаге:
- создаётся SQLAlchemy‑движок для выбранной БД (в данном примере — локальный SQLite‑файл);
- вызывается `create_tables(engine)`, чтобы гарантированно создать все необходимые таблицы (`articles`, `clusters`, `entities`).

In [6]:
db_path = "sqlite:///./test.db"
engine = get_engine(db_path)
create_tables(engine)

Session = sessionmaker(bind=engine)
session = Session()
session.query(Article).delete()
session.commit()

#### Настройка датасета, создание и добавление статей в БД:

In [13]:
N = 100  

# конверт в список словарей
train_list = train_data.select(range(N)).to_dict()

for i in range(N):
    row = {k: train_list[k][i] for k in train_list}  
    art = Article(
        external_id=f"rfn_{i}",
        title=row["title"] if row["title"] else "no title",
        text=row["body"] if row["body"] else "",
        published_at=datetime.now(timezone.utc),
        processed=True,
    )
    session.add(art)

session.commit()
session.close()
print(f"Inserted {N} articles into local DB.")


Inserted 100 articles into local DB.


***
### Создание экземпляра `EventClustering` и Запуск кластеризации (`run_once`)

Здесь создаётся объект кластеризатора с заданной конфигурацией:
- `db_url` — указывает, к какой базе данных подключаться;
- `embedding_model` — какую модель эмбеддингов использовать (по умолчанию — мультиязычная);
- `min_cluster_size` — минимальный размер кластера для HDBSCAN;
- `use_hdbscan` — флаг, использовать ли HDBSCAN или сразу переходить к KMeans.

Вызов `ec.run_once(limit=50)` запускает полный цикл:

1. выбираются не больше 50 статей из таблицы `articles`, которые ещё не были кластеризованы;
2. для каждой статьи строится эмбеддинг (заголовок + текст);
3. эмбеддинги группируются в кластеры с помощью HDBSCAN или KMeans;
4. информация о кластерах записывается в таблицу `clusters`;
5. у статей обновляются поля `cluster_id` и `clustered_at`.

В логах можно увидеть:
- сколько статей было обработано;
- какой алгоритм использовался;
- какие метки кластеров получились и сколько статей попало в каждый кластер;
- значение silhouette‑метрики (если оно удалось корректно посчитать).

In [11]:
ec = EventClustering(db_url=db_path, use_hdbscan=False)
ec.run_once(limit=50)

INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cpu
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: paraphrase-multilingual-MiniLM-L12-v2
INFO:event_clustering:Fetched 20 unclustered articles
INFO:event_clustering:run_once: fetched 20 articles to cluster
INFO:event_clustering:run_once: prepared 20 combined texts
INFO:event_clustering:run_once: got embeddings shape=(20, 384)
INFO:event_clustering:run_once: clustering done, algo=kmeans, labels=[0, 1, 2, 3], counts=[2, 5, 6, 7]
INFO:event_clustering:run_once: silhouette score = 0.0956
INFO:event_clustering:persist_clusters: labels present = [2, 1, 0, 3]
INFO:event_clustering:persist_clusters: created cluster id=1, lab=2, size=6, top_terms=['title', 'предлагать', 'выпуск', 'правоурмийский', 'русоловый', 'начало', 'концентрат', 'медный', 'прибыль', 'акционер']
INFO:event_clustering:persist_clusters: created cluster id=2, lab=1, size=5, top_terms=['уровень', 'рубль', 'iii', 'title',

In [15]:
ec = EventClustering(db_url=db_path)  
ec.run_once(limit=500) #остановил, так как слишком много времени требует на 100 статей

INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cpu
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: paraphrase-multilingual-MiniLM-L12-v2
INFO:event_clustering:Fetched 100 unclustered articles
INFO:event_clustering:run_once: fetched 100 articles to cluster
INFO:event_clustering:run_once: prepared 100 combined texts
INFO:event_clustering:run_once: got embeddings shape=(100, 384)
INFO:event_clustering:run_once: clustering done, algo=hdbscan, labels=[-1, 0, 1], counts=[79, 9, 12]
INFO:event_clustering:run_once: silhouette score = -0.0022
INFO:event_clustering:persist_clusters: labels present = [-1, 0, 1]


KeyboardInterrupt: 

***
### Просмотр результатов кластеризации

Здесь с помощью SQLAlchemy выполняются несколько простых запросов к БД:

- общее количество статей;
- сколько из них получили `cluster_id` (то есть были отнесены к кластерам);
- сколько кластеров создано в таблице `clusters`;
- распределение статей по кластерам (например, сколько статей в каждом кластере).

Этот шаг нужен для визуальной проверки:
- что кластеры действительно создаются;
- что статьи корректно привязаны к кластерам;
- какие темы приблизительно отражают сформированные кластеры (можно дополнительно вывести `top_terms`).

In [12]:
db_path = "sqlite:///./test.db"
engine = get_engine(db_path)
Session = sessionmaker(bind=engine)

session = Session()

print("=== Общая статистика ===")
total = session.query(Article).count()
with_text = session.query(Article).filter(Article.text != "").count()
clustered = session.query(Article).filter(Article.cluster_id.isnot(None)).count()
print(f"Всего статей: {total}")
print(f"С непустым текстом: {with_text}")
print(f"С присвоенным cluster_id: {clustered}")

print("\n=== Распределение по кластерам ===")
clusters = session.query(Article.cluster_id).distinct().all()
print("Уникальные cluster_id:", clusters)

for (cid,) in clusters:
    count = session.query(Article).filter(Article.cluster_id == cid).count()
    print(f"cluster_id={cid}: {count} статей")

print("\n=== Кластеры ===")
all_clusters = session.query(Cluster).all()
print(f"Всего кластеров в таблице Cluster: {len(all_clusters)}")
for cl in all_clusters:
    print(f"- Cluster id={cl.id}, size={cl.size}, algo={cl.algorithm}, top_terms={cl.top_terms}")

session.close()

=== Общая статистика ===
Всего статей: 20
С непустым text: 20
С присвоенным cluster_id: 20

=== Распределение по кластерам ===
Уникальные cluster_id: [(1,), (2,), (3,), (4,)]
cluster_id=1: 6 статей
cluster_id=2: 5 статей
cluster_id=3: 2 статей
cluster_id=4: 7 статей

=== Кластеры ===
Всего кластеров в таблице Cluster: 4
- Cluster id=1, size=6, algo=kmeans, top_terms=['title', 'предлагать', 'выпуск', 'правоурмийский', 'русоловый', 'начало', 'концентрат', 'медный', 'прибыль', 'акционер']
- Cluster id=2, size=5, algo=kmeans, top_terms=['уровень', 'рубль', 'iii', 'title', 'квартал', 'тыс', 'год', 'рост', 'свыше', 'инвест']
- Cluster id=3, size=2, algo=kmeans, top_terms=['нефть', 'сфера', 'развитие', 'проникать', 'понятный', 'пока', 'следовать', 'слово', 'набиуллина', 'технология']
- Cluster id=4, size=7, algo=kmeans, top_terms=['трлн', 'title', 'акция', 'становиться', 'бкс', 'иностранный', 'год', 'биржа', 'заблокировать', 'определять']


In [10]:
from sqlalchemy import func
from event_clustering import get_engine, Article, Cluster

db_path = "sqlite:///./test.db"
engine = get_engine(db_path)
Session = sessionmaker(bind=engine)
session = Session()

print("=== Общая статистика по Article ===")
total = session.query(func.count(Article.id)).scalar()
clustered = session.query(func.count(Article.id)).filter(Article.cluster_id.isnot(None)).scalar()
noise = session.query(func.count(Article.id)).filter(Article.cluster_id.is_(None), Article.clustered_at.isnot(None)).scalar()
never_touched = session.query(func.count(Article.id)).filter(Article.cluster_id.is_(None), Article.clustered_at.is_(None)).scalar()

print(f"Всего статей: {total}")
print(f"С cluster_id != NULL: {clustered}")
print(f"Отмечены как 'шум' (cluster_id=NULL, clustered_at NOT NULL): {noise}")
print(f"Никогда не кластеризовались: {never_touched}")

print("\n=== Кластеры ===")
clusters = session.query(Cluster).all()
print(f"Всего кластеров: {len(clusters)}")
for cl in clusters[:10]:
    print(f"id={cl.id}, algo={cl.algorithm}, size={cl.size}, top_terms={cl.top_terms}")

session.close()

=== Общая статистика по Article ===
Всего статей: 20
С cluster_id != NULL: 0
Отмечены как 'шум' (cluster_id=NULL, clustered_at NOT NULL): 20
Никогда не кластеризовались: 0

=== Кластеры ===
Всего кластеров: 4
id=1, algo=kmeans, size=6, top_terms=['title', 'предлагать', 'выпуск', 'правоурмийский', 'русоловый', 'начало', 'концентрат', 'медный', 'прибыль', 'акционер']
id=2, algo=kmeans, size=5, top_terms=['уровень', 'рубль', 'iii', 'title', 'квартал', 'тыс', 'год', 'рост', 'свыше', 'инвест']
id=3, algo=kmeans, size=2, top_terms=['нефть', 'сфера', 'развитие', 'проникать', 'понятный', 'пока', 'следовать', 'слово', 'набиуллина', 'технология']
id=4, algo=kmeans, size=7, top_terms=['трлн', 'title', 'акция', 'становиться', 'бкс', 'иностранный', 'год', 'биржа', 'заблокировать', 'определять']


In [6]:
engine = get_engine("sqlite:///./test.db")
Session = sessionmaker(bind=engine)
session = Session()

# Сколько кластеризованных статей
print("Всего статей:", session.query(Article).count())
print("Кластеризованных (cluster_id IS NOT NULL):", session.query(Article).filter(Article.cluster_id != None).count())

# Показать все кластеры и их метаданные
clusters = session.query(Cluster).all()
for c in clusters:
    print("Cluster id:", c.id, "size:", c.size, "top_terms:", c.top_terms, "algo:", c.algorithm)

# Показать статьи по кластерам
for c in clusters:
    arts = session.query(Article).filter(Article.cluster_id == c.id).all()
    print(f"\n== Cluster {c.id} ({c.size}): top_terms={c.top_terms} ==")
    for a in arts:
        print(" -", a.id, a.title or "(no title)", "|", (a.text[:120] + "...") if a.text else "")

session.close()


Всего статей: 9
Кластеризованных (cluster_id IS NOT NULL): 9
Cluster id: 1 size: 4 top_terms: ['bank', 'ceo', 'announced', 'bankruptcy', 'file', 'for', 'the', 'public', 'company', 'startup'] algo: kmeans
Cluster id: 2 size: 4 top_terms: ['the', 'bank', 'central', 'rate', 'thousand', 'change', 'capital', 'demanding', 'rallied', 'downtown'] algo: kmeans
Cluster id: 3 size: 5 top_terms: ['bank', 'ceo', 'региональный', 'банкротство', 'announced', 'file', 'bankruptcy', 'startup', 'company', 'intention'] algo: kmeans
Cluster id: 4 size: 4 top_terms: ['центробанк', 'ставка', 'tax', 'candidate', 'debate', 'election', 'heating', 'infrastructure', 'mayor', 'local'] algo: kmeans

== Cluster 1 (4): top_terms=['bank', 'ceo', 'announced', 'bankruptcy', 'file', 'for', 'the', 'public', 'company', 'startup'] ==

== Cluster 2 (4): top_terms=['the', 'bank', 'central', 'rate', 'thousand', 'change', 'capital', 'demanding', 'rallied', 'downtown'] ==

== Cluster 3 (5): top_terms=['bank', 'ceo', 'региональный